In [ ]:
import os
import sys
import dspy
from edgar.entity.core import Company
from edgar.core import set_identity

# --- 1. ENVIRONMENT SETUP (Fixing the Deno/Jupyter Issue) ---
deno_path = os.path.expanduser("~/.deno/bin")
if deno_path not in os.environ.get("PATH", ""):
    os.environ["PATH"] += os.pathsep + deno_path

# (Optional) If running in a Jupyter Notebook, this verifies Deno is found
# !deno --version


# --- 2. SEC DATA RETRIEVAL ---
set_identity("Quantitative Research quant_nlp@financialanalysis.com")

def retrieve_sec_context(ticker: str, form_type: str = "10-K") -> str:
    """
    Retrieves the latest filing and extracts Risk Factors and MD&A,
    compiling them with specific headers for the AI to parse.
    """
    company = Company(ticker)
    filings = company.get_filings(form=form_type)
    filing_obj = filings.latest().obj()
    
    risk_factors_text = filing_obj.risk_factors
    mda_text = filing_obj.management_discussion
    
    # These exact headers will be referenced in the DSPy Signature
    context_string = f"--- ITEM 1A: RISK FACTORS ---\n{risk_factors_text}\n\n"
    context_string += f"--- ITEM 7: MANAGEMENT'S DISCUSSION AND ANALYSIS ---\n{mda_text}"
    
    return context_string


# --- 3. DSPy LANGUAGE MODELS ---
# Ensure your OPENAI_API_KEY is set in your environment variables
main_lm = dspy.LM("openai/gpt-4o")
cheap_lm = dspy.LM("openai/gpt-4o-mini")
dspy.configure(lm=main_lm)


# --- 4. RLM SIGNATURE ---
class SECRiskAnalysisSignature(dspy.Signature):
    """
    Programmatically analyze massive SEC filing documents to extract critical risk 
    factors, forward-looking statements, and management evaluations based on the provided query.
    """
    document: str = dspy.InputField(desc="Text from SEC EDGAR filings. It is explicitly divided by the exact text headers '--- ITEM 1A: RISK FACTORS ---' and '--- ITEM 7: MANAGEMENT\\'S DISCUSSION AND ANALYSIS ---'. Use these exact headers to split the text.")
    query: str = dspy.InputField(desc="The specific financial, regulatory, or risk-based question to answer.")
    
    analysis_narrative: str = dspy.OutputField(desc="A comprehensive, highly detailed narrative analysis synthesizing data from the document.")
    critical_risks_identified: list[str] = dspy.OutputField(desc="A Python list of strings detailing the top distinct material risks identified.")
    hallucination_check: bool = dspy.OutputField(desc="Returns True only if the system is fully confident all facts are explicitly present in the text.")





# --- 6. INSTANTIATE AND RUN RLM ---
# Notice: No extra extraction tools are passed here, forcing the AI to use the `document` input
sec_analyzer_rlm = dspy.RLM(
    SECRiskAnalysisSignature, 
    max_iterations=15,
    sub_lm=cheap_lm,
    verbose=True
)

# Fetch the data
print("Fetching SEC data for AAPL...")
massive_sec_data = retrieve_sec_context("AAPL")
print(f"Data retrieved! Length: {len(massive_sec_data)} characters.\n")

complex_query = "What specific cybersecurity risks and resulting material cost increases are highlighted in the MD&A and Risk Factors, and how do they relate to the 2018 SEC guidance?"

# Execute the pipeline
print("Starting RLM analysis loop...")
result = sec_analyzer_rlm(document=massive_sec_data, query=complex_query)

print("\n=== FINAL RESULTS ===")
print("NARRATIVE ANALYSIS:\n", result.analysis_narrative)
print("\nIDENTIFIED RISKS:\n", result.critical_risks_identified)
print("\nHALLUCINATION CHECK PASSED:\n", result.hallucination_check)

Fetching SEC data for AAPL...
Data retrieved! Length: 86266 characters.

Starting RLM analysis loop...


2026/03/09 01:26:30 INFO dspy.predict.rlm: RLM iteration 1/15
Reasoning: Let's start by exploring the `document` to understand its structure and identify the relevant pieces associated with 'ITEM 1A: RISK FACTORS' and 'ITEM 7: MANAGEMENT'S DISCUSSION AND ANALYSIS'. We'll split the document based on these headers to isolate the sections. This will allow us to focus on the parts of the document relevant to the `query`, which asks about cybersecurity risks and cost increases highlighted in these sections.

For the first step, I'll split the document into sections based on the provided headers to prepare the data for further analysis.
Code:
```python
# Splitting the document based on the specified item headers
sections = document.split('--- ')
risk_factors_section = next((section for section in sections if section.startswith('ITEM 1A: RISK FACTORS')), None)
mda_section = next((section for section in sections if section.startswith("ITEM 7: MANAGEMENT'S DISCUSSION AND ANALYSIS")), None)

# P


=== FINAL RESULTS ===
NARRATIVE ANALYSIS:
 The analysis of the 'Risk Factors' and MD&A sections reveals several critical cybersecurity risks that could result in material cost increases for the Company:

1. The Company faces significant risks from ransomware and other cybersecurity attacks that could disrupt business operations, leading to increased costs for prevention and recovery measures.
2. There are vulnerabilities associated with information technology system failures and network disruptions, posing significant risks to business continuity and financial performance.
3. Unauthorized access to or loss of confidential information can result in substantial costs due to legal liabilities, reputational damage, and remedial actions.
4. Regular malicious attacks necessitate increased expenses for enhanced cybersecurity defenses, including potential regulatory fines and litigation costs.
5. The limitations of insurance coverage in fully addressing data security risks imply potential une